In [ ]:
######基础版MLP 0.27 调整优化器和学习率调度器
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt  

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 调整模型以匹配输入形状 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # 编码器：
        self.encoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.Tanh(),
            nn.Linear(16, 8),
            nn.Tanh(),
            nn.Linear(8, 1),
        )
        
        # 解码器：
        self.decoder = nn.Sequential(
            nn.Linear(1, 8),
            nn.Tanh(),
            nn.Linear(8, 16),
            nn.Tanh(),
            nn.Linear(16, 8),
        )
        
        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        if quantized.dim() == 0:
            quantized = quantized.unsqueeze(0)
        binary = torch.zeros(quantized.numel(), self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary.squeeze(0)

    def binary_to_quantized(self, binary):
        if binary.dim() == 1:
            binary = binary.unsqueeze(0)
        quantized = torch.zeros(binary.shape[0], dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized.squeeze(0)

    def dequantize(self, x_quantized):
        if x_quantized.dim() == 0:
            x_quantized = x_quantized.unsqueeze(0)
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)
        batch_size = x.shape[0]
        
        # 编码：(batch_size, 1, 8) -> (batch_size, 1)
        x_encoded = self.encoder(x.squeeze(1))  
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # 量化
        x_quantized = self.quantizer(x_encoded)
        x_binary = self.quantized_to_binary(x_quantized)
        x_quantized_recovered = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized = self.dequantize(x_quantized_recovered)
        x_dequantized = x_dequantized.unsqueeze(1)  
        
        # 解码：(batch_size, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized)
        x_output = x_decoded.unsqueeze(1) 
        
        return x_output, x_quantized, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 新增：绘制损失曲线函数
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    """
    绘制训练损失曲线
    :param loss_history: 存储每轮平均损失
    :param title: 图表标题
    """
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    
    data = np.load(data_path)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    
    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 损失函数和优化器
    criterion = nn.L1Loss()
    optimizer = optim.SGD(codec.parameters(), lr=0.005, momentum=0.85,nesterov=True)
    #optimizer = optim.Adam(codec.parameters(), lr=0.0005)
    #scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=196, eta_min=0.00005)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50,T_mult=2,eta_min=0.00005)
    
    # 训练循环
    num_epochs = 100
    codec.train()
    
    # 新增：用于存储每轮平均损失的列表
    loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            # 将数据移到设备上
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            # 梯度裁剪
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            # 每100个批次
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        # 更新学习率
        scheduler.step()
        
        # 计算 epoch 的平均损失
        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)  # 新增：保存每轮损失
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    # 新增：训练结束后绘制损失曲线
    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")
    
    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    # 计算平均测试损失
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")




In [ ]:
######1 MLP换为CNN 0.239330  调优化器-0.239345  batch_size=128-0.239416  调学习率为1e-4-0.2398150    LeakyReLU-0.238931

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt  

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),  # (batch, 1, 1)
            nn.Tanh()
        )
        
        # 解码器：使用一维卷积转置
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),  # (batch, 1, 8)
            nn.Tanh()
        )
        
        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        """
        将量化后的张量转换为二进制张量。
        输入: (batch_size,) 一维张量
        输出: (batch_size, bitwidth) 二维张量
        """
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            # 从最低位到最高位提取
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        """
        将二进制张量转换回量化后的张量。
        输入: (batch_size, bitwidth) 二维张量
        输出: (batch_size,) 一维张量
        """
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        """
        将量化值解量化为原始范围的值。
        输入: (batch_size,) 一维张量
        输出: (batch_size,) 一维张量
        """
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)
        
        # 编码：(batch_size, 1, 8) -> (batch_size, 1, 1)
        x_encoded = self.encoder(x)  
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # 确保量化前的张量是一维的 (batch_size,)
        # .view(-1) 会将任何形状（如 (batch_size, 1, 1) 或 (batch_size, 1)）安全地展平为一维
        x_encoded_1d = x_encoded.view(-1)
        
        # 量化
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        
        # 将一维张量恢复为解码器需要的三维形状 (batch_size, 1, 1)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        
        # 解码：(batch_size, 1, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized_3d)
        
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    data = np.load(data_path)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 损失函数和优化器
    criterion = nn.L1Loss()
    #optimizer = optim.SGD(codec.parameters(), lr=0.005, momentum=0.85,nesterov=True)
    optimizer = optim.AdamW(codec.parameters(),lr=1e-3,weight_decay=1e-4 )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50,T_mult=2,eta_min=0.00005) #50/30
    
    # 训练循环
    num_epochs = 100
    codec.train()
    
    loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")
    
    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")




